# SIE Quickstart

Three functions - `encode`, `score`, `extract` - running on your laptop or a free Colab GPU.

This notebook installs the SIE server, starts it in the background, and walks through the full API.

## 1. Install and start the server

Works on your laptop (Mac or Linux, CPU is fine) or a free Colab GPU. The `[local]` extra pins the transformers version the embedding and reranking models need.

On Colab, expect a few minutes: SIE pins torch 2.9, so pip replaces Colab's preinstalled PyTorch.

> **Note:** SIE requires Python 3.12. Colab's current default runtime is Python 3.12, so no action is needed there. Locally, run this notebook on a Python 3.12 kernel.

In [ ]:
%pip install -q "sie-server[local]" sie-sdk

In [ ]:
import subprocess, time, urllib.request

# Start the server in the background (defaults: host 0.0.0.0, port 8080)
proc = subprocess.Popen(
    ["sie-server", "serve", "--port", "8080"],
    stdout=open("/tmp/sie-server.log", "w"),
    stderr=subprocess.STDOUT,
)

# Wait for the server to be ready (/healthz answers before any model loads)
for i in range(120):
    try:
        urllib.request.urlopen("http://localhost:8080/healthz")
        print(f"Server ready (took ~{i}s)")
        break
    except Exception:
        time.sleep(1)
else:
    print("Server did not start - check /tmp/sie-server.log")

## 2. Encode

Generate dense embeddings with a 400M-parameter model. The model downloads on first use (a few minutes depending on bandwidth), then subsequent calls are fast.

On machines without an NVIDIA GPU, SIE automatically falls back to a slower sentence-transformers execution path for this model.

In [ ]:
from sie_sdk import SIEClient
from sie_sdk.types import Item

client = SIEClient("http://localhost:8080")

result = client.encode("NovaSearch/stella_en_400M_v5", Item(text="Hello world"))
print(f"Shape: {result['dense'].shape}")   # (1024,)
print(f"First 5 values: {result['dense'][:5]}")

## 3. Score

Rerank documents by relevance using a cross-encoder.

In [ ]:
scores = client.score(
    "BAAI/bge-reranker-v2-m3",
    Item(text="What is machine learning?"),
    [Item(text="ML learns from data."), Item(text="The weather is sunny.")]
)

for s in scores["scores"]:
    print(f"  rank {s['rank']}: {s['score']:.3f} - {s['item_id']}")

## 4. Extract

Zero-shot named entity recognition - no training data needed.

In [ ]:
result = client.extract(
    "urchade/gliner_multi-v2.1",
    Item(text="Tim Cook is the CEO of Apple."),
    labels=["person", "organization"]
)

for e in result["entities"]:
    print(f"  {e['label']}: {e['text']} ({e['score']:.2f})")

## Next steps

- [API reference](https://superlinked.com/docs/reference/sdk) - full SDK documentation
- [All models](https://superlinked.com/models) - 85+ supported models
- [Deployment guide](https://superlinked.com/docs/deployment/docker) - run in production with Docker or Kubernetes
- [GitHub](https://github.com/superlinked/sie) - star the repo if this was useful